# Lesson 2.5: Bayes' Theorem

Find the chance of really being sick after a positive test (1% prevalence, 99% sensitivity, 95% specificity), by Bayes' formula and by counting 10,000 people.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A screening test for a disease is 99% sensitive (it flags 99% of sick people) and 95% specific (it correctly clears 95% of healthy people). The disease affects 1% of people. A patient tests positive and asks: what is the chance I am really sick? Can Bayes' theorem answer this, and does counting 10,000 people give the same answer?

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: The test's numbers

These are the lesson's numbers. Sensitivity is P(positive given sick); the false alarm rate is 1 minus the specificity. Write them down once and reuse them.

In [1]:
prevalence = 0.01      # P(sick)  # Store how common the disease is: 1% of people are sick. This is the prior, the chance before any test.
sensitivity = 0.99     # P(positive | sick)  # Store the sensitivity: the chance of a positive test when sick.
specificity = 0.95     # P(negative | healthy)  # Store the specificity: the chance of a negative test when healthy.
false_alarm = 1 - specificity  # A healthy person tests positive with probability 1 minus the specificity. This is the false alarm rate.

print('P(sick)                =', prevalence)  # Print the prevalence so we can read it back.
print('P(positive | sick)     =', sensitivity)  # Print the sensitivity.
print('P(positive | healthy)  =', round(false_alarm, 2))  # Print the false alarm rate, rounded to avoid decimal noise.

P(sick)                = 0.01
P(positive | sick)     = 0.99
P(positive | healthy)  = 0.05


**What this means.** We have three facts: 1% of people are sick, 99% of the sick test positive, and 5% of the healthy test positive by mistake. Everything else is derived from these three numbers, so we state them once and reuse them.

## Step 2: Bayes' formula

P(sick | positive) = P(positive | sick) x P(sick) / P(positive), where P(positive) adds up both ways of testing positive: sick people and healthy people with a false alarm.

In [2]:
p_pos = sensitivity * prevalence + false_alarm * (1 - prevalence)  # Add up both ways of testing positive: sick people who are caught, plus healthy people with a false alarm. This is the total probability of a positive.
ppv = sensitivity * prevalence / p_pos  # Bayes' formula: the share of all positives that are truly sick, which is P(positive | sick) x P(sick) / P(positive).

print(f'P(positive)         = {p_pos:.4f}')  # Print the overall chance of a positive test.
print(f'P(sick | positive)  = {ppv:.4f}')  # Print the chance of being sick given a positive.

P(positive)         = 0.0594
P(sick | positive)  = 0.1667


**What this means.** About 5.94% of all people test positive, and only 0.1667 (about 16.7%) of the people who test positive are actually sick. A positive test raises the chance from 1% to 17%, but most positives are still false alarms.

## Step 3: Check with 10,000 people

Count people instead of multiplying probabilities. Of 10,000 people, 100 are sick, and 5% of the 9,900 healthy people get a false positive. Only the positives matter for the question.

In [3]:
people = 10_000  # Imagine 10,000 people.
sick = round(people * prevalence)  # About 1% of them are sick.
healthy = people - sick  # The rest are healthy.

true_pos = round(sick * sensitivity)  # Of the sick people, 99% test positive (true positives).
false_pos = round(healthy * false_alarm)  # Of the healthy people, 5% test positive (false positives).

print('sick:', sick, ' healthy:', healthy)  # Print how many are sick and healthy.
print('true positives:', true_pos, ' false positives:', false_pos)  # Print the true and false positive counts.
print(f'{true_pos} / {true_pos + false_pos} = {true_pos / (true_pos + false_pos):.4f}')  # Among everyone who tested positive, print the share who are sick.

sick: 100  healthy: 9900
true positives: 99  false positives: 495
99 / 594 = 0.1667


**What this means.** Of 10,000 people, 100 are sick and 9,900 are healthy. There are 99 true positives and 495 false positives, so 99 / 594 = 0.1667. Counting gives the same answer as the formula, and it shows why: the healthy group is so big that its 5% errors outnumber the sick.

## Step 4: Why the prior matters

Keep the same test and change only how common the disease is. The same positive result means very different things for a rare and a common disease.

In [4]:
for prior in [0.001, 0.01, 0.10, 0.30, 0.50]:  # Try several prevalences, from very rare to very common.
    pos = sensitivity * prior + false_alarm * (1 - prior)  # Recompute the total chance of a positive for each prevalence.
    print(f'prevalence {prior:>5.1%}: P(sick | positive) = {sensitivity * prior / pos:.2%}')  # Print the chance of being sick given a positive, as a percentage.

prevalence  0.1%: P(sick | positive) = 1.94%
prevalence  1.0%: P(sick | positive) = 16.67%
prevalence 10.0%: P(sick | positive) = 68.75%
prevalence 30.0%: P(sick | positive) = 89.46%
prevalence 50.0%: P(sick | positive) = 95.19%


**What this means.** With the same test, a positive result means a 1.94% chance of being sick when the disease affects 0.1% of people, 16.67% at 1%, 68.75% at 10%, 89.46% at 30% and 95.19% at 50%. The meaning of a positive test depends heavily on how common the disease is.

## So, what is the answer?

A person who tests positive has about a 16.7% chance (99 in 594) of being sick, even though the test is 99% sensitive, because the disease is rare and false alarms from the large healthy group outnumber true cases. The caution: the answer depends on the prevalence, so the same test result means more in a group where the disease is common.

## Now you try

An email filter flags 95% of spam and also wrongly flags 3% of legitimate email. A company finds that 20% of its incoming mail is spam. An employee sees a flagged message and asks how likely it is truly spam. The company also wonders what happens in a quieter inbox where only 2% is spam.

**Data:** P(spam) = 0.20, P(flagged | spam) = 0.95, P(flagged | legitimate) = 0.03. Also try spam shares of 0.02 and 0.50.

1. Find P(flagged), the overall chance a message is flagged, when 20% is spam.
2. Find P(spam | flagged) for spam shares of 2%, 20% and 50%.
3. Check the 20% case by counting 10,000 messages.

*Hint:* Add the two ways to be flagged (spam caught plus legitimate mistakes), then divide the spam-caught part by that total. Wrap the formula in a function so you can reuse it for each prior.

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
def posterior(prior, sens=0.95, false_alarm=0.03):
    p_flag = sens * prior + false_alarm * (1 - prior)
    return sens * prior / p_flag

print(f'P(flagged) when 20% is spam = {0.95 * 0.20 + 0.03 * 0.80:.4f}')
for prior in [0.02, 0.20, 0.50]:
    print(f'spam share {prior:>4.0%}: P(spam | flagged) = {posterior(prior):.4f}')

n = 10_000
spam = round(n * 0.20)
print('check with counts:', round(spam * 0.95), 'true flags,', round((n - spam) * 0.03), 'false flags')
print(f'P(spam | flagged) by counting = {1900 / 2140:.4f}')

P(flagged) when 20% is spam = 0.2140
spam share   2%: P(spam | flagged) = 0.3926
spam share  20%: P(spam | flagged) = 0.8879
spam share  50%: P(spam | flagged) = 0.9694
check with counts: 1900 true flags, 240 false flags
P(spam | flagged) by counting = 0.8879


**Answer.** When 20% is spam, P(flagged) = 0.2140 and P(spam | flagged) = 0.8879. The result changes with the spam share: 0.3926 at 2%, 0.8879 at 20% and 0.9694 at 50%. Counting 10,000 messages gives 1,900 true flags and 240 false flags, and 1900 / 2140 = 0.8879 again. A flag is much more trustworthy when spam is common.

## The same question in R, SPSS and Stata

**R**

```r
prev <- 0.01; sens <- 0.99; spec <- 0.95  # Store prevalence, sensitivity and specificity on one line, separated by semicolons.
p_pos <- sens * prev + (1 - spec) * (1 - prev)  # Total chance of a positive test: caught sick people plus false alarms.
sens * prev / p_pos                 # Bayes' formula  # Bayes' formula for P(sick | positive).

sick <- 10000 * prev                 # 100  # Number of sick people in 10,000: 100.
true_pos  <- sick * sens             # 99  # True positives: sick people the test catches.
false_pos <- (10000 - sick) * (1 - spec)   # 495  # False positives: healthy people the test wrongly flags.
true_pos / (true_pos + false_pos)  # Share of positives that are sick, by counting.
```

How to read it: Both calculations print 0.1667. The first is Bayes' formula (p_pos is 0.0594); the second is 99 true positives out of 99 + 495 = 594 positives. Same answer, two routes.

**SPSS** (Transform > Compute Variable)

```
DATA LIST FREE / dummy.
BEGIN DATA
1
END DATA.
COMPUTE p_pos = 0.99*0.01 + 0.05*0.99.
COMPUTE ppv = 0.99*0.01 / p_pos.
COMPUTE true_pos = 10000*0.01*0.99.
COMPUTE false_pos = 10000*0.99*0.05.
COMPUTE ppv_counts = true_pos / (true_pos + false_pos).
FORMATS p_pos ppv ppv_counts (F8.4).
LIST VARIABLES=p_pos ppv true_pos false_pos ppv_counts.
```

How to read it: In the LIST output look at p_pos (0.0594), ppv and ppv_counts (both 0.1667), and true_pos and false_pos (99 and 495). The formula and the counts agree.

**Stata**

```
display "P(positive) = " (0.99*0.01 + 0.05*0.99)
display "Bayes: " (0.99*0.01 / (0.99*0.01 + 0.05*0.99))
display "true positives: " (10000*0.01*0.99)
display "false positives: " (10000*0.99*0.05)
display "counts: " (99 / (99 + 495))
```

How to read it: The five lines print .0594, .1666667, 99, 495 and .1666667. Stata drops the leading zero, so .1666667 is 0.1667. The formula and the counts agree.